# Sumteo — record one camera on Colab's GPU (Day 6, replay mode)

ARCHITECTURE §2.11. A laptop can't run four cameras live, so each camera's pipeline run is **recorded once** here and
the dashboard replays it with no models loaded.

- **One camera per run.** The recording run is that camera's solo test: check the printed levels and events.
- **Only processed output is kept:** the blurred + overlaid video, the state/event timeline, blurred snapshots and a
  manifest. No raw frames.
- **Keep the repo layout** (`data/raw/…`, `models/…`): paths are part of the config hash, and a recording whose hash
  differs from the Mac's config refuses to replay ("recording is stale, re-record").
- **crowd_07 is CroHD (CC BY-NC-SA 3.0):** its recording stays non-commercial, with credit; keep it in `data/` (git-ignored).
- Never paste keys or tokens into this notebook. Commit it **without outputs**.

Runtime → Change runtime type → **T4 GPU** first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the code (default: a zip from the Mac)

The recording must come from the **same commit** as the Mac that will replay it. On the Mac, after committing:

```
git archive --format=zip HEAD -o sumteo_src.zip
```

Upload `sumteo_src.zip` with the next cell. (Cloning instead works only for a pushed branch of a public repo; never
paste a token into this notebook.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # sumteo_src.zip
!mkdir -p /content/sumteo && unzip -q -o sumteo_src.zip -d /content/sumteo

In [ ]:
# Optional instead of the zip: clone a pushed branch of a public repo
# REPO_URL = "https://github.com/sanjardev23/sumteo.git"
# !git clone --depth 1 --branch develop {REPO_URL} /content/sumteo

In [ ]:
%cd /content/sumteo
!pip install -q -e .

## 2. Inputs — clips and model weights at the repo paths

Use **the same files as the Mac** (copy them from the Mac's `data/raw/` and `models/`): the manifest stores their
sha256, and the Mac refuses a recording made with a different file at the same path.

Clips: `data/raw/crowd_02.mp4`, `crowd_03.mp4`, `crowd_04.mp4`, `crowd_07.mp4` (only the one you record is needed).
Weights: `models/yolo26n.pt`, `models/dmcount_shb.pth`, `models/dmcount_nwpu.pth`, `models/face_detection_yunet_2023mar.onnx`.

Easiest: put them once in Google Drive under `MyDrive/sumteo/data/raw/` and `MyDrive/sumteo/models/`, then run the
Drive cell. Or upload them directly with the cell after it.

In [ ]:
# From Google Drive (same folder layout under MyDrive/sumteo)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/sumteo"
!mkdir -p data/raw models
!cp -n {DRIVE}/data/raw/*.mp4 data/raw/ 2>/dev/null; cp -n {DRIVE}/models/* models/ 2>/dev/null; ls -la data/raw models

In [ ]:
# Or upload directly (pick the clip for this camera, then the weights)
from google.colab import files
import shutil, os
os.makedirs("data/raw", exist_ok=True); os.makedirs("models", exist_ok=True)
for name in files.upload():
    shutil.move(name, ("data/raw/" if name.endswith(".mp4") else "models/") + name)
!ls -la data/raw models

## 3. Record one camera

Cameras: `crossing_demo` (crowd_07), `crowd_demo` (crowd_02), `test_crowd_03`, `test_crowd_04`.
Watch the printed levels and events: this run is the camera's solo test.

In [ ]:
CAMERA = "crossing_demo"
!python scripts/record_camera.py --camera configs/cameras/{CAMERA}.yaml --device cuda --overwrite

In [ ]:
import json, glob
for path in glob.glob("data/recordings/*/manifest.json"):
    print(path, json.dumps(json.load(open(path)), indent=2))

## 4. Bring the recording back to the Mac

Download the zip (or copy it to Drive), then on the Mac unzip it into `data/recordings/` and start the API:

```
python scripts/serve.py --replay data/recordings/<camera id> [more recordings…] [--cameras configs/cameras/crowd_demo.yaml]
```

In [ ]:
from sumteo.config import load_config
camera_id = load_config(f"configs/cameras/{CAMERA}.yaml")[1].id
!cd data/recordings && zip -q -r /content/{camera_id}.zip {camera_id}
from google.colab import files
files.download(f"/content/{camera_id}.zip")